# Duty cycle experiment A — how does duty cycle change performance of a trained model?

**Question:** for a *fixed* trained U-Net and the three classical baselines, how do recall, false-positive rate and pixel AP change as the RFI duty cycle changes?

**Design (controlled sweep, no training):**
- Recordings are generated with `simulate_recording`. For one scene seed, frequency, start/stop, phase and antenna gains stay identical; only `duty_cycle` (and the peak `strength`) change. Each seed is therefore its own control (paired design).
- Duty cycle is changed at fixed period, so the pulse gets longer as duty rises. Strength is **peak** (on-state) power, so average power rises with duty.
- Factors: family × peak strength × duty cycle × scene seed. Radar uses its own log-spaced duty range and strengths, close to the starter data.
- The U-Net keeps its validation-calibrated threshold (`calibration.json`) — no retuning per duty level.

**Reading the metrics:** recall and FPR do not depend on how many pixels are RFI; precision and AP do (more positives at higher duty raise them on their own), so compare *methods* at the same duty rather than reading the absolute AP trend. Uncertainty is a 95% bootstrap over scene seeds (the recording is the unit, not the pixel).

**Hypothesis to check:** the spectral-kurtosis estimator is reported to be blind to strong on/off RFI near 50% duty and to continuous (100%) signals, so `sk` recall should dip there while the U-Net stays flatter.

## 1. Settings

In [ ]:
import os
import sys
from pathlib import Path
from cs163_rfi import paths  # folders come from .env at the repo root
MODEL_RUN_DIR = Path(os.environ.get("RFI_MODEL_RUN_DIR", paths.RESULTS_DIR /
                     "runs/grid_20261022T083531Z_pmuteget_unet_b8_0b2beafa/bce_lr0.001"))  # best grid trial (val AP 0.635)
MODEL_CHECKPOINT_DIR = Path(os.environ.get("RFI_MODEL_CHECKPOINT_DIR", paths.CHECKPOINTS_ROOT /
                            "prince/runs/grid_20261022T083531Z_pmuteget_unet_b8_0b2beafa/bce_lr0.001"))  # best.pt for MODEL_RUN_DIR
OUT_DIR = Path(os.environ.get("RFI_DUTY_A_DIR", paths.RESULTS_DIR / "duty_cycle/A_eval_sweep"))
SWEEP = {  # family: (duty cycles, peak strengths)
    "switched_carrier": ((0.1, 0.25, 0.5, 0.75, 0.9, 1.0), (0.1, 0.5, 2.0)),
    "bpsk":             ((0.1, 0.25, 0.5, 0.75, 0.9, 1.0), (0.1, 0.5, 2.0)),
    "radar":            ((0.001, 0.003, 0.01, 0.03, 0.1, 0.3), (1.0, 5.0, 20.0)),
}
SCENE_SEEDS = tuple(range(500001, 500009))  # starter dataset seeds are 96000-120039
METHODS = ("sk", "bandpass", "sk-bandpass")
BASELINE_FAR = 1e-4  # same as the grid notebook
AP_BINS = 4096

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.metrics import average_precision_score
from IPython.display import display
from cs163_rfi import simulate_recording, flag_rfi, score_mask

torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", 4)))
plt.rcParams.update({"font.size": 11, "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "· output:", OUT_DIR)

## 2. Load the trained U-Net and its calibrated threshold

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.ReLU(inplace=True))
    def forward(self, x): return self.layers(x)

class SegmentationModel(nn.Module):
    """Same two-level U-Net as RFI_unet_team.ipynb."""
    def __init__(self, base=8, use_skips=True):
        super().__init__()
        self.use_skips = use_skips
        self.enc1 = DoubleConv(1, base)
        self.enc2 = DoubleConv(base, 2*base)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(2*base, 4*base)
        self.up2 = nn.ConvTranspose2d(4*base, 2*base, 2, stride=2)
        self.dec2 = DoubleConv(4*base if use_skips else 2*base, 2*base)
        self.up1 = nn.ConvTranspose2d(2*base, base, 2, stride=2)
        self.dec1 = DoubleConv(2*base if use_skips else base, base)
        self.output = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        z = self.bottleneck(self.pool(e2))
        z = self.up2(z)
        z = self.dec2(torch.cat((z, e2), dim=1) if self.use_skips else z)
        z = self.up1(z)
        z = self.dec1(torch.cat((z, e1), dim=1) if self.use_skips else z)
        return self.output(z)

def histogram_ap(positive, negative):
    """Pixel AP from quantized-score histograms (counts are sample weights)."""
    if positive.sum() == 0: return np.nan
    bins = len(positive)
    labels = np.r_[np.ones(bins), np.zeros(bins)]
    scores = np.tile(np.arange(bins), 2)
    weights = np.r_[positive, negative]
    keep = weights > 0
    return float(average_precision_score(labels[keep], scores[keep], sample_weight=weights[keep]))

In [ ]:
best = torch.load(MODEL_CHECKPOINT_DIR / "best.pt", map_location="cpu", weights_only=False)
calibration = json.loads((MODEL_RUN_DIR / "calibration.json").read_text())
CONFIG, NORM = best["config"], best["normalization"]
model = SegmentationModel(CONFIG["base_channels"], CONFIG["architecture"] == "unet").to(device).eval()
model.load_state_dict(best["model_state_dict"])
CUT, CAL_BINS = calibration["quantized_cut"], calibration["probability_bins"]
print(f"{CONFIG['architecture']}/{CONFIG['base_channels']} {CONFIG['loss']} lr={CONFIG['learning_rate']:g}, "
      f"epoch {best['epoch']}, val AP {best['validation_ap']:.4f}; threshold p >= {calibration['probability_lower_bound']:.4f}")

@torch.inference_mode()
def predict_recording(power):
    """All ten antennas in one batch; returns probabilities with axes (time, antenna, channel)."""
    x = (power[..., 0].transpose(1, 0, 2).astype(np.float32) - NORM["mean"]) / NORM["std"]
    p = model(torch.from_numpy(x)[:, None].to(device)).sigmoid()[:, 0].cpu().numpy()
    return p.transpose(1, 0, 2)

## 3. Run the sweep
One row per recording × method holds TP/FP/FN/TN summed over the ten antennas. U-Net score histograms are pooled per (family, strength, duty) for pixel AP.

In [ ]:
rows, hists = [], {}
total = sum(len(SCENE_SEEDS) * len(d) * len(s) for d, s in SWEEP.values())
done, started = 0, time.perf_counter()
for family, (duties, strengths) in SWEEP.items():
    for seed in SCENE_SEEDS:
        for strength in strengths:
            for duty in duties:
                rec = simulate_recording(seed=seed, rfi=family, strength=strength, duty_cycle=duty)
                power, truth = rec["power"], rec["rfi_mask"]
                p = predict_recording(power)
                flags = {m: flag_rfi(power, method=m, far=BASELINE_FAR) for m in METHODS}
                flags["unet"] = (np.floor(p * (CAL_BINS - 1)) >= CUT)[..., None]
                for method, mask in flags.items():
                    s = score_mask(mask, truth)
                    rows.append(dict(family=family, seed=seed, strength=strength, duty_cycle=duty,
                                     method=method, tp=s["tp"], fp=s["fp"], fn=s["fn"], tn=s["tn"],
                                     labelled_fraction=float(truth.mean())))
                q = np.floor(p * (AP_BINS - 1)).astype(np.int64)
                t = truth[..., 0]
                h = hists.setdefault((family, strength, duty), np.zeros((2, AP_BINS), np.int64))
                h[0] += np.bincount(q[t], minlength=AP_BINS)
                h[1] += np.bincount(q[~t], minlength=AP_BINS)
                done += 1
        print(f"{family} seed {seed}: {done}/{total} recordings, {time.perf_counter() - started:.0f} s", flush=True)
results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "per_recording.csv", index=False)
pd.DataFrame(list(hists), columns=["family", "strength", "duty_cycle"]).to_csv(OUT_DIR / "unet_histograms_index.csv")
np.save(OUT_DIR / "unet_histograms.npy", np.stack(list(hists.values())))  # rows match the index CSV; [positive, negative]

## 4. Summary tables

In [ ]:
def pooled(group, rng, n_boot=1000):
    """Pooled recall/FPR/precision, with recall CI from resampling scene seeds."""
    tp, fp, fn, tn = (group[c].to_numpy() for c in ("tp", "fp", "fn", "tn"))
    idx = rng.integers(0, len(tp), (n_boot, len(tp)))
    boot = tp[idx].sum(1) / np.maximum((tp + fn)[idx].sum(1), 1)
    ratio = lambda a, b: a / b if b else np.nan
    return dict(recall=ratio(tp.sum(), tp.sum() + fn.sum()),
                recall_lo=np.quantile(boot, .025), recall_hi=np.quantile(boot, .975),
                false_positive_rate=ratio(fp.sum(), fp.sum() + tn.sum()),
                precision=ratio(tp.sum(), tp.sum() + fp.sum()),
                labelled_fraction=group.labelled_fraction.mean(), recordings=len(group))

rng = np.random.default_rng(0)
keys = ["family", "strength", "duty_cycle", "method"]
summary = pd.DataFrame([dict(zip(keys, k), **pooled(g, rng)) for k, g in results.groupby(keys)])
unet_ap = pd.DataFrame([dict(family=f, strength=s, duty_cycle=d, unet_pixel_ap=histogram_ap(*h))
                        for (f, s, d), h in hists.items()])
summary.to_csv(OUT_DIR / "summary.csv", index=False)
unet_ap.to_csv(OUT_DIR / "unet_pixel_ap.csv", index=False)
with pd.option_context("display.max_rows", 50, "display.float_format", "{:.4f}".format):
    display(summary.pivot_table(index=["family", "strength", "duty_cycle"], columns="method", values="recall"))
    display(unet_ap.pivot_table(index=["family", "duty_cycle"], columns="strength", values="unet_pixel_ap"))

## 5. Plots

In [ ]:
COLORS = {"unet": "#2a78d6", "sk": "#eb6834", "bandpass": "#1baf7a", "sk-bandpass": "#eda100"}

def plot_by_method(metric, ylabel, filename, band=None, sharey=True):
    fig, axes = plt.subplots(len(SWEEP), 3, figsize=(13, 3.3 * len(SWEEP)), sharey=sharey, layout="constrained")
    for i, (family, (duties, strengths)) in enumerate(SWEEP.items()):
        for j, strength in enumerate(strengths):
            ax = axes[i, j]
            for method, color in COLORS.items():
                d = summary.query("family == @family and strength == @strength and method == @method").sort_values("duty_cycle")
                ax.plot(d.duty_cycle, d[metric], marker="o", ms=6, lw=2, color=color, label=method)
                if band:
                    ax.fill_between(d.duty_cycle, d[band[0]], d[band[1]], color=color, alpha=.15, lw=0)
            if family == "radar": ax.set_xscale("log")
            ax.set_title(f"{family} · peak strength {strength:g}")
            ax.grid(alpha=.2)
        axes[i, 0].set_ylabel(ylabel)
    for ax in axes[-1]: ax.set_xlabel("Duty cycle")
    axes[0, -1].legend(frameon=False)
    fig.savefig(OUT_DIR / filename, dpi=200)
    plt.show(); plt.close(fig)

plot_by_method("recall", "Recall (95% CI over seeds)", "recall_vs_duty.png", band=("recall_lo", "recall_hi"))
plot_by_method("false_positive_rate", "False-positive rate", "fpr_vs_duty.png", sharey=False)

In [ ]:
STRENGTH_SHADES = ("#86b6ef", "#2a78d6", "#104281")  # light -> dark = weak -> strong
fig, axes = plt.subplots(1, len(SWEEP), figsize=(13, 3.6), sharey=True, layout="constrained")
for ax, (family, (duties, strengths)) in zip(axes, SWEEP.items()):
    for strength, color in zip(strengths, STRENGTH_SHADES):
        d = unet_ap.query("family == @family and strength == @strength").sort_values("duty_cycle")
        ax.plot(d.duty_cycle, d.unet_pixel_ap, marker="o", ms=6, lw=2, color=color, label=f"strength {strength:g}")
    if family == "radar": ax.set_xscale("log")
    ax.set(title=family, xlabel="Duty cycle")
    ax.legend(frameon=False); ax.grid(alpha=.2)
axes[0].set_ylabel("U-Net pixel AP")
fig.savefig(OUT_DIR / "unet_ap_vs_duty.png", dpi=200)
plt.show(); plt.close(fig)

## 6. How to read the results
- **Recall vs duty** is the main result. A flat U-Net curve means the model is insensitive to duty cycle at that strength; a dip shows where it misses.
- Compare the `sk` curve near duty 0.5 and 1.0 with the U-Net (SK blind-spot hypothesis).
- **FPR** should stay near the clean-calibrated level; if it rises with duty, the model or baseline is flagging neighbours of strong RFI.
- **AP** rises with duty partly because more pixels are RFI; use it to compare strengths at the same duty, not as an absolute trend.
- Files in `OUT_DIR`: `per_recording.csv`, `summary.csv`, `unet_pixel_ap.csv` and the three PNGs.